# Archived team notebook
See [workflow and limitations](README.md) before running. Outputs and session metadata have been cleared. Supply your own data paths and checkpoints. These notebooks were not rerun during packaging.

In [ ]:
%%bash
set -euxo pipefail

# 基本工具
apt -y install -qq aria2

# micromamba 建 Python 3.10 環境（A1111 官方相容）
cd /content
if [ ! -x /content/bin/micromamba ]; then
  curl -sL https://micro.mamba.pm/api/micromamba/linux-64/latest -o micromamba.tar.bz2
  tar -xjvf micromamba.tar.bz2 -C /content bin/micromamba
fi
/content/bin/micromamba create -y -p /content/py310 -c conda-forge python=3.10 pip
/content/py310/bin/python -V

# 取得 A1111 專案
if [ ! -d /content/stable-diffusion-webui ]; then
  git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui /content/stable-diffusion-webui
fi
cd /content/stable-diffusion-webui
# Existing checkout is preserved; select a tested commit manually.

# 準備模型/LoRA 資料夾
mkdir -p models/Stable-diffusion models/Lora
echo "✅ Step A OK"

In [ ]:
%%bash
set -euxo pipefail
cd /content/stable-diffusion-webui
/content/py310/bin/python -m pip install -U pip wheel
/content/py310/bin/python -m pip install -r requirements_versions.txt --prefer-binary 2>&1 | tee /content/pip_install.log
tail -n 60 /content/pip_install.log || true
echo "✅ Step B OK"

In [ ]:
%%bash
set -euxo pipefail
MODEL_DIR="/content/stable-diffusion-webui/models/Stable-diffusion"
FILE="realisticVisionV20_v20.safetensors"
mkdir -p "$MODEL_DIR"

# 正確用法：-d 指定資料夾，-o 只有檔名，避免路徑重複
aria2c -x16 -s16 \
  -d "$MODEL_DIR" \
  -o "$FILE" \
  "https://huggingface.co/ckpt/realistic-vision-v20/resolve/main/realisticVisionV20_v20.safetensors?download=true" || true

# 若曾誤存到 /content/content/...，就搬回來
if [ -f "/content/content/stable-diffusion-webui/models/Stable-diffusion/$FILE" ]; then
  mv -f "/content/content/stable-diffusion-webui/models/Stable-diffusion/$FILE" "$MODEL_DIR/"
fi

ls -lh "$MODEL_DIR"
echo "✅ Step C-1 OK（若列表有 realisticVisionV20_v20.safetensors 就成功）"

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%%bash
set -euo pipefail
SRC="/content/drive/MyDrive/lora_crack_dataset/lora_crack_datasetV1.safetensors"   # ←改成你的檔名
DST="/content/stable-diffusion-webui/models/Lora"
mkdir -p "$DST"

if [ -f "$SRC" ]; then
  cp -n "$SRC" "$DST/"
  echo "✅ 已複製：$(basename "$SRC") -> $DST"
else
  echo "⚠️ 找不到檔案：$SRC"
  echo "📂 試列出 lora_dataset："
  ls -lh "/content/drive/MyDrive/lora_crack_dataset" || true
fi

echo "📂 目前 Lora 資料夾："
ls -lh "$DST" || true

In [ ]:
from safetensors.torch import load_file
import os, sys

path = "/content/drive/MyDrive/lora_crack_dataset/lora_crack_datasetV1.safetensors"  # 換成你的路徑
sd = load_file(path)
keys = list(sd.keys())[:30]
print("檔案：", os.path.basename(path))
print("tensor 數量：", len(sd))
print("前幾個 key：")
for k in keys: print("  ", k)

def guess_format(keys):
    text = " ".join(keys)
    if any(k.startswith("lora_unet_") or "lora_te_" in k for k in keys):
        return "➡️ 看起來是 A1111 / kohya 風格（可直接給 A1111 用）"
    if "lora.down.weight" in text or "lora_A.weight" in text or "lora_B.bias" in text:
        return "➡️ 看起來是 PEFT/diffusers 風格（A1111 不吃，需要轉檔）"
    return "➡️ 格式不明，可能需要轉檔再試"
print(guess_format(keys))


In [ ]:
# === 參數 ===
SRC = "/content/drive/MyDrive/lora_crack_dataset/lora_crack_datasetV1.safetensors"  # 你的 LoRA (舊格式)
DST = "/content/stable-diffusion-webui/models/Lora/lora_crack_datasetV1_webui.safetensors"  # 輸出 (新格式)
RANK = 8
ALPHA = 32

from safetensors.torch import load_file, save_file
import os, torch

sd = load_file(SRC)
new_sd = {}
def rename_key(k: str):
    if not k.endswith(".weight"):
        return None
    if k.startswith("base_model.model."):
        body = k[len("base_model.model."):]
        body = body.replace(".", "_")
        body = "lora_unet_" + body
        body = body.replace("_lora_A_weight", ".lora_down.weight")
        body = body.replace("_lora_B_weight", ".lora_up.weight")
        return body
    return None

for k, v in sd.items():
    nk = rename_key(k)
    if nk:
        new_sd[nk] = v

metadata = {
    "ss_network_module": "networks.lora",
    "ss_network_dim": str(RANK),
    "ss_network_alpha": str(ALPHA),
    "format": "kohya_lora",
}
os.makedirs(os.path.dirname(DST), exist_ok=True)
save_file(new_sd, DST, metadata=metadata)

print("✅ 已轉換：", DST)
print("轉換 tensor 數量：", len(new_sd))
print("範例 key：", list(new_sd.keys())[:5])


Historical public-tunnel launch cell omitted from this public copy. It launched WebUI with insecure extension access and no authentication. See README.md for launch limitations. The model setup and conversion cells above remain historical code; conversion compatibility is not verified.

Historical public-tunnel launch cell omitted from this public copy. It launched WebUI with insecure extension access and no authentication. See README.md for launch limitations. The model setup and conversion cells above remain historical code; conversion compatibility is not verified.